# Uniformity of the survey as epochs accumulate (rolling cadence, WFD without DDF)

- creation date : 2026-10-09
- based on : `03_N_Per_Season.ipynb` (same WFD footprint, same SQL constraint, same custom-metric approach)

## Overview and outputs

Notebook `03_N_Per_Season` shows the rolling cadence **season by season**. Static cosmology (large-scale structure, weak lensing, 3x2pt) instead needs a survey that is **uniform in number of visits / depth as the data accumulate**. There is no ready-made MAF metric for this, so this notebook builds the cumulative maps and quantifies their uniformity.

Two definitions of 'after N epochs' are computed in one pass over the database:

- **time-based** (`n_time`): visits taken before `t0 + N years` (`t0 = SURVEY_START_MJD`, 1 year = 365.25 d). This is the 'less than N years' definition.
- **season-based** (`n_season`): visits taken in the first N observing seasons *of each sky position* (`calc_season < N`, the season start depends on RA). Because every patch of sky has its own season phase, the time-based maps contain an RA-dependent offset of up to about one season; the season-based maps remove it and isolate the effect of the rolling cadence.

Steps:
1. Define a custom MAF metric (`CumulativeVisitsMetric`) returning, for each sky position, the cumulative number of visits and the cumulative coadded depth in one band, for N = 1 ... 10.
2. Run it on the WFD pixels (no DDF) of the baseline simulation.
3. Map the cumulative number of visits for each epoch (absolute and normalised to its median).
4. Quantify the uniformity versus epoch (relative RMS, robust sigma, 5-95 percentile spread, fraction of pixels within 10% of the median) and compare time-based and season-based definitions.
5. Same for the coadded depth in one band.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (path set in `opsim_fname`).

**Outputs**: arrays and tables go to `DATA06_DDF_RollingUniformity/`; figures (PNG and PDF) go to `FIG06_DDF_RollingUniformity/`.

## 0. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
from IPython.display import display

import rubin_sim.maf as maf
from rubin_sim.maf import BaseMetric

from rubin_scheduler.scheduler.utils import CurrentAreaMap
from rubin_scheduler.utils import SURVEY_START_MJD, calc_season

## 1. Configuration

- `N_YEARS`: number of cumulative epochs (1 to 10 years, or 1 to 10 seasons).
- `MIN_EXP_TIME`: visits shorter than this are dropped (near-sun twilight visits), as in notebook 03.
- `DEPTH_FILTER`: band used for the coadded depth.
- `REL_MIN`, `REL_MAX`: colour range of the maps normalised to their median; `DM5_RANGE`: half colour range of the depth maps (mag).
- Output directories `DATA_DIR` / `FIG_DIR` are created.

In [ ]:
# OpSim database (same as notebook 03)
opsim_fname = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
runName = os.path.split(opsim_fname)[-1].replace(".db", "")
print(opsim_fname)
print(runName)

NSIDE = 64  # HEALPix resolution
N_YEARS = 10  # number of cumulative epochs
YEAR = 365.25  # days
MIN_EXP_TIME = 16  # s
DEPTH_FILTER = "i"  # band for the coadded depth
REL_MIN, REL_MAX = 0.5, 1.5
DM5_RANGE = 0.3  # mag

# Output directories: arrays/tables go to DATA_DIR, all figures go to FIG_DIR
DATA_DIR = "DATA06_DDF_RollingUniformity"
FIG_DIR = "FIG06_DDF_RollingUniformity"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

KIND_LABEL = {
    "n_time": "time-based",
    "n_season": "season-based",
    "m5_time": "time-based",
    "m5_season": "season-based",
}
WHEN = {
    "n_time": "visits before t0 + {n} yr",
    "n_season": "visits in the first {n} seasons",
    "m5_time": "depth before t0 + {n} yr",
    "m5_season": "depth in the first {n} seasons",
}

## 2. Custom MAF metric

`CumulativeVisitsMetric` works on one sky position. It keeps the visits longer than `min_exp_time`, computes the season of each visit with `calc_season` (same function as notebook 03), and returns a dictionary of arrays of length `n_years`:

- `n_time[k-1]`, `n_season[k-1]`: number of visits for epoch `k` (time-based / season-based).
- `m5_time[k-1]`, `m5_season[k-1]`: coadded 5-sigma depth in `depth_filter`, `1.25 log10(sum 10^(0.8 m5))`; NaN if there is no visit in that band yet.

A position that has no visit at all keeps masked in MAF; it is treated as 0 visits (and no depth) in the next sections.

In [ ]:
# allow re-running the cell: remove a previous registration of the metric
if "__main__.CumulativeVisitsMetric" in BaseMetric.registry:
    del BaseMetric.registry["__main__.CumulativeVisitsMetric"]


class CumulativeVisitsMetric(BaseMetric):
    """Cumulative number of visits and coadded depth after N years / N seasons.

    Parameters
    ----------
    n_years : `int`
        Number of cumulative epochs (1 ... n_years).
    depth_filter : `str`
        Band used for the coadded depth.
    min_exp_time : `float`
        Minimum exposure time (s) to count a visit (drops twilight visits).
    mjd_start : `float` or None
        Survey start (mjd). Default None uses SURVEY_START_MJD.
    """

    def __init__(
        self,
        n_years=10,
        depth_filter="i",
        min_exp_time=16,
        mjd_start=None,
        mjd_col="observationStartMJD",
        exp_time_col="visitExposureTime",
        m5_col="fiveSigmaDepth",
        filter_col="filter",
        metric_name="CumulativeVisits",
        **kwargs,
    ):
        if mjd_start is None:
            mjd_start = SURVEY_START_MJD
        self.mjd_start = mjd_start
        self.n_years = n_years
        self.depth_filter = depth_filter
        self.min_exp_time = min_exp_time
        self.mjd_col = mjd_col
        self.exp_time_col = exp_time_col
        self.m5_col = m5_col
        self.filter_col = filter_col
        super().__init__(
            col=[mjd_col, exp_time_col, m5_col, filter_col],
            units="#",
            metric_name=metric_name,
            metric_dtype="object",
            **kwargs,
        )

    @staticmethod
    def _coadd(m5):
        if len(m5) == 0:
            return np.nan
        return 1.25 * np.log10(np.sum(10.0 ** (0.8 * m5)))

    def run(self, data_slice, slice_point):
        n = self.n_years
        out = {
            "n_time": np.zeros(n),
            "n_season": np.zeros(n),
            "m5_time": np.full(n, np.nan),
            "m5_season": np.full(n, np.nan),
        }
        data = data_slice[data_slice[self.exp_time_col] > self.min_exp_time]
        if len(data) == 0:
            return out
        mjd = data[self.mjd_col]
        # SlicePoints ra is in radians: convert to degrees for calc_season
        season = calc_season(np.degrees(slice_point["ra"]), mjd, self.mjd_start)
        m5 = data[self.m5_col]
        in_band = data[self.filter_col] == self.depth_filter
        for k in range(1, n + 1):
            sel_t = mjd < self.mjd_start + k * YEAR
            sel_s = season < k
            out["n_time"][k - 1] = np.count_nonzero(sel_t)
            out["n_season"][k - 1] = np.count_nonzero(sel_s)
            out["m5_time"][k - 1] = self._coadd(m5[sel_t & in_band])
            out["m5_season"][k - 1] = self._coadd(m5[sel_s & in_band])
        return out

## 3. Metric set-up on the WFD footprint (no DDF)

Same selection as notebook 03:
- WFD pixels from the scheduler labels (`lowdust`, `euclid_overlap`, `LMC_SMC`, `virgo`), used as a `HealpixSubsetSlicer`.
- The SQL constraint removes DDF, ToO and twilight visits, so only regular survey visits are counted.

A single constraint is used (all epochs are computed inside the metric), so the database is read only once.

In [ ]:
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()

wfdhpid = np.where(
    (labels == "lowdust") | (labels == "euclid_overlap") | (labels == "LMC_SMC") | (labels == "virgo")
)[0]
wfd_slicer = maf.HealpixSubsetSlicer(nside=NSIDE, hpid=wfdhpid)

constraint = (
    "scheduler_note not like '%dd%' and scheduler_note not like '%too%' and scheduler_note not like '%twi%'"
)
label = "WFD (no DDF)"

metric = CumulativeVisitsMetric(n_years=N_YEARS, depth_filter=DEPTH_FILTER, min_exp_time=MIN_EXP_TIME)
bundle = maf.MetricBundle(metric, wfd_slicer, constraint, info_label=label, run_name=runName)
g = maf.MetricBundleGroup({"cum": bundle}, opsim_fname, out_dir=DATA_DIR, results_db=None, verbose=True)

## 4. Run the metric and extract the cumulative maps

After running, the object array `bundle.metric_values` is unpacked into four arrays of shape `(N_YEARS, n_pixels)` stored in the dictionary `cum`. `HPID` gives the HEALPix index of each WFD pixel. Pixels with no visit get 0 visits and NaN depth. The arrays are saved in `DATA_DIR`.

In [ ]:
g.run_all()

In [ ]:
HPID = np.asarray(wfd_slicer.slice_points["sid"])
npix = len(HPID)

mv = bundle.metric_values
mask = np.ma.getmaskarray(mv)

cum = {
    "n_time": np.zeros((N_YEARS, npix)),
    "n_season": np.zeros((N_YEARS, npix)),
    "m5_time": np.full((N_YEARS, npix), np.nan),
    "m5_season": np.full((N_YEARS, npix), np.nan),
}
n_filled = 0
for ip in range(npix):
    if mask[ip]:
        continue
    v = mv.data[ip]
    if not isinstance(v, dict):
        continue
    n_filled += 1
    for key in cum:
        cum[key][:, ip] = v[key]

print(f"WFD pixels: {npix}, pixels with at least one visit: {n_filled}")
np.savez(os.path.join(DATA_DIR, f"cumulative_maps_{runName}.npz"), hpid=HPID, **cum)

## 5. Uniformity statistics versus epoch

For each epoch and each definition, over all WFD pixels (pixels with no visit counted as 0):

- `rms/mean`: standard deviation divided by the mean of the number of visits.
- `robust_sigma/median`: `1.4826 * MAD / median`, insensitive to the tails.
- `(p95-p5)/median`: spread between the 5th and 95th percentiles.
- `frac_within_10pc`: fraction of pixels within +/-10% of the median.
- `frac_zero`: fraction of pixels without any visit.

For the depth (`DEPTH_FILTER` band): standard deviation, robust sigma and 5-95 percentile spread in magnitudes, and the fraction of pixels within 0.05 mag of the median. Tables are saved as CSV in `DATA_DIR`.

In [ ]:
def stats_counts(x):
    med = np.median(x)
    mean = np.mean(x)
    p5, p95 = np.percentile(x, [5, 95])
    robust = 1.4826 * np.median(np.abs(x - med))
    ok = med > 0
    return {
        "median": med,
        "mean": mean,
        "rms/mean": np.std(x) / mean if mean > 0 else np.nan,
        "robust_sigma/median": robust / med if ok else np.nan,
        "(p95-p5)/median": (p95 - p5) / med if ok else np.nan,
        "frac_within_10pc": np.mean(np.abs(x / med - 1) < 0.10) if ok else np.nan,
        "frac_zero": np.mean(x == 0),
    }


def stats_depth(m5):
    ok = np.isfinite(m5)
    if ok.sum() == 0:
        return {
            "median_m5": np.nan,
            "std": np.nan,
            "robust_sigma": np.nan,
            "p95-p5": np.nan,
            "frac_within_0.05mag": np.nan,
            "frac_with_data": 0.0,
        }
    y = m5[ok]
    med = np.median(y)
    p5, p95 = np.percentile(y, [5, 95])
    return {
        "median_m5": med,
        "std": np.std(y),
        "robust_sigma": 1.4826 * np.median(np.abs(y - med)),
        "p95-p5": p95 - p5,
        "frac_within_0.05mag": np.mean(np.abs(y - med) < 0.05),
        "frac_with_data": ok.mean(),
    }


rows = []
for kind in ["n_time", "n_season"]:
    for k in range(N_YEARS):
        r = stats_counts(cum[kind][k])
        r.update(kind=kind, n=k + 1)
        rows.append(r)
df_counts = pd.DataFrame(rows)

rows = []
for kind in ["m5_time", "m5_season"]:
    for k in range(N_YEARS):
        r = stats_depth(cum[kind][k])
        r.update(kind=kind, n=k + 1)
        rows.append(r)
df_depth = pd.DataFrame(rows)

df_counts.to_csv(os.path.join(DATA_DIR, f"uniformity_nvisits_{runName}.csv"), index=False)
df_depth.to_csv(os.path.join(DATA_DIR, f"uniformity_depth_{DEPTH_FILTER}_{runName}.csv"), index=False)

display(df_counts.round(3))
display(df_depth.round(3))

## 6. Sky maps of the cumulative number of visits

`to_fullsky` puts the WFD pixel values on a full HEALPix map (other pixels are `hp.UNSEEN`). For each epoch, `plot_year` draws two Mollweide maps and saves them (PNG + PDF) in `FIG_DIR`:

- left: absolute cumulative number of visits, same colour scale for all epochs of a given definition (0 to the 99.5th percentile of the last epoch), so that the accumulation is visible;
- right: the same map divided by its median (colour range `REL_MIN` - `REL_MAX`), which shows the non-uniformity at that epoch independently of the total number of visits.

In [ ]:
CMAP_DIV = plt.get_cmap("RdBu_r")


def to_fullsky(values):
    m = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)
    good = np.isfinite(values)
    m[HPID[good]] = values[good]
    return m


def save_fig(fig, name):
    base = os.path.join(FIG_DIR, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")


def plot_year(kind, k, vmax_abs):
    n = k + 1
    x = cum[kind][k]
    med = np.median(x)
    fig = plt.figure(figsize=(14, 4.5))
    hp.mollview(
        to_fullsky(x),
        fig=fig.number,
        sub=(1, 2, 1),
        min=0,
        max=vmax_abs,
        unit="visits",
        title=f"{KIND_LABEL[kind]}: {WHEN[kind].format(n=n)} (median = {med:.0f})",
    )
    hp.mollview(
        to_fullsky(x / max(med, 1.0)),
        fig=fig.number,
        sub=(1, 2, 2),
        min=REL_MIN,
        max=REL_MAX,
        cmap=CMAP_DIV,
        unit="N / median(N)",
        title="normalised to the median",
    )
    fig.suptitle(f"{runName} {label}")
    save_fig(fig, f"nvisits_{kind}_{n:02d}")
    plt.show()

### Figures 1-10 - Time-based: visits taken in less than N years (N = 1 ... 10)

Files `nvisits_n_time_01` ... `nvisits_n_time_10` in `FIG_DIR`.

In [ ]:
kind = "n_time"
vmax_abs = np.percentile(cum[kind][-1], 99.5)
for k in range(N_YEARS):
    plot_year(kind, k, vmax_abs)

### Figures 11-20 - Season-based: visits in the first N seasons of each position (N = 1 ... 10)

Same maps, but the epoch is counted in observing seasons of each sky position instead of calendar time. Comparing with the time-based maps separates the RA-dependent season-phase effect from the rolling-cadence effect. Files `nvisits_n_season_01` ... `nvisits_n_season_10`.

In [ ]:
kind = "n_season"
vmax_abs = np.percentile(cum[kind][-1], 99.5)
for k in range(N_YEARS):
    plot_year(kind, k, vmax_abs)

## 7. Mosaics of the normalised maps

All ten epochs on one figure (2 x 5 panels) for a quick look at how the non-uniformity evolves: number of visits divided by its median (`REL_MIN` - `REL_MAX`), and coadded depth minus its median (+/- `DM5_RANGE` mag). Panels have no colour bar; the colour range is given in the figure title.

In [ ]:
def plot_mosaic(kind, normalise, vmin, vmax, title, fname):
    fig = plt.figure(figsize=(22, 8))
    for k in range(N_YEARS):
        x = cum[kind][k]
        if normalise:
            v = x / max(np.median(x), 1.0)
        else:
            v = x - np.nanmedian(x)
        hp.mollview(
            to_fullsky(v),
            fig=fig.number,
            sub=(2, 5, k + 1),
            min=vmin,
            max=vmax,
            cmap=CMAP_DIV,
            cbar=False,
            title=f"N = {k + 1}",
        )
    fig.suptitle(f"{runName} {label} - {KIND_LABEL[kind]} - {title}", fontsize=14)
    save_fig(fig, fname)
    plt.show()


for kind in ["n_time", "n_season"]:
    plot_mosaic(
        kind,
        True,
        REL_MIN,
        REL_MAX,
        f"N visits / median, colour range {REL_MIN} - {REL_MAX}",
        f"mosaic_{kind}",
    )

for kind in ["m5_time", "m5_season"]:
    plot_mosaic(
        kind,
        False,
        -DM5_RANGE,
        DM5_RANGE,
        f"{DEPTH_FILTER}-band coadded depth - median (mag), colour range +/-{DM5_RANGE}",
        f"mosaic_{kind}_{DEPTH_FILTER}",
    )

## 8. Uniformity versus epoch

### Figure - Uniformity indicators for the number of visits (`uniformity_vs_epoch_nvisits`)

Four indicators from the table of section 5 as a function of the epoch N, for the time-based and the season-based definitions. A rolling cadence that is compatible with a uniform final survey shows indicators that decrease with N; the season-based curves show the rolling-cadence effect alone.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
panels = [
    ("rms/mean", "RMS / mean"),
    ("robust_sigma/median", "robust sigma / median"),
    ("(p95-p5)/median", "(p95 - p5) / median"),
    ("frac_within_10pc", "fraction of pixels within +/-10% of median"),
]
for ax, (col, ttl) in zip(axs.ravel(), panels):
    for kind in ["n_time", "n_season"]:
        d = df_counts[df_counts["kind"] == kind]
        ax.plot(d["n"], d[col], "o-", label=KIND_LABEL[kind])
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
axs[0, 0].legend()
for ax in axs[1]:
    ax.set_xlabel("epoch N (years or seasons)")
fig.suptitle(f"{runName} {label} - number of visits")
fig.tight_layout()
save_fig(fig, "uniformity_vs_epoch_nvisits")
plt.show()

### Figure - Uniformity indicators for the coadded depth (`uniformity_vs_epoch_depth`)

Standard deviation, robust sigma and 5-95 percentile spread of the coadded depth in the `DEPTH_FILTER` band (mag), and fraction of pixels within 0.05 mag of the median, versus epoch.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
panels = [
    ("std", "std of m5 (mag)"),
    ("robust_sigma", "robust sigma of m5 (mag)"),
    ("p95-p5", "p95 - p5 of m5 (mag)"),
    ("frac_within_0.05mag", "fraction of pixels within 0.05 mag of median"),
]
for ax, (col, ttl) in zip(axs.ravel(), panels):
    for kind in ["m5_time", "m5_season"]:
        d = df_depth[df_depth["kind"] == kind]
        ax.plot(d["n"], d[col], "o-", label=KIND_LABEL[kind])
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
axs[0, 0].legend()
for ax in axs[1]:
    ax.set_xlabel("epoch N (years or seasons)")
fig.suptitle(f"{runName} {label} - {DEPTH_FILTER}-band coadded depth")
fig.tight_layout()
save_fig(fig, f"uniformity_vs_epoch_depth_{DEPTH_FILTER}")
plt.show()

## 9. Distribution of the normalised number of visits

### Figure - Histograms of N / median(N) for a few epochs (`hist_nvisits_normalised`)

One panel per definition. A uniform survey gives a narrow peak at 1; a rolling cadence in progress gives a broad or multi-peaked distribution that narrows as the epochs accumulate.

In [ ]:
epochs_to_show = [1, 2, 3, 5, 10]
bins = np.linspace(0, 2, 81)
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, kind in zip(axs, ["n_time", "n_season"]):
    for n in epochs_to_show:
        if n > N_YEARS:
            continue
        x = cum[kind][n - 1]
        ax.hist(x / max(np.median(x), 1.0), bins=bins, histtype="step", lw=1.5, label=f"N = {n}")
    ax.set_xlabel("N visits / median")
    ax.set_title(KIND_LABEL[kind])
    ax.grid(True, alpha=0.3)
    ax.legend()
axs[0].set_ylabel("number of pixels")
fig.suptitle(f"{runName} {label}")
fig.tight_layout()
save_fig(fig, "hist_nvisits_normalised")
plt.show()

## Notes on reading the results

- **Time-based, N = 1**: the survey has just started, some pixels may have few visits or none (`frac_zero`); this epoch mostly shows the survey start-up, not the rolling cadence.
- **Time-based vs season-based**: for the time-based epoch, a patch whose season starts just before `t0 + N yr` has one fewer completed season than a patch whose season ends just after it; this adds an RA-dependent term of order 1/N that is not related to the rolling cadence. The season-based definition removes it.
- The cumulative **depth** reacts more slowly than the number of visits (depth scales as 1.25 log10 of the number of visits), so the same relative non-uniformity in N gives a smaller non-uniformity in magnitudes.
- Thresholds used for the 'uniform' fractions (10% in N, 0.05 mag in m5) are arbitrary; change them in `stats_counts` / `stats_depth` to match the requirement of the cosmological probe under study.